# Model Evaluation: Linear Regression vs XGBoost

**Objective**: Technical evaluation of regression models for delivery time prediction

**Models**:
- Linear Regression (baseline)
- XGBoost Regressor (champion candidate)

**Evaluation Focus**:
- Performance comparison (MAE, RMSE, R²)
- Residual diagnostics
- Segment-level stress testing
- Evidence-based analysis (no premature conclusions)

## 1. Data Loading and Preparation

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import warnings

lr_pred_train = lr_model.predict(X_train)
lr_pred_val = lr_model.predict(X_val)
lr_pred_test = lr_model.predict(X_test)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [6]:
# Load dataset
df = pd.read_csv('Dataset/model_ready_data.csv')

# Temporal ordering
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.sort_values('order_date').reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['order_date'].min()} to {df['order_date'].max()}")
print(f"\nTarget variable: delivery_time_hours")
print(f"Target stats: mean={df['delivery_time_hours'].mean():.2f}h, std={df['delivery_time_hours'].std():.2f}h")

Dataset shape: (69926, 17)
Date range: 2025-11-06 00:00:00 to 2026-01-05 00:00:00

Target variable: delivery_time_hours
Target stats: mean=9.16h, std=6.04h


### Time-Aware Split (70/15/15)

In [7]:
# Define target and features
TARGET = 'delivery_time_hours'
DROP_COLS = ['order_date', 'route_id', 'destination_city', TARGET]

# Time-ordered split
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

# Prepare features and target
X_train = train_df.drop(columns=DROP_COLS)
y_train = train_df[TARGET]

X_val = val_df.drop(columns=DROP_COLS)
y_val = val_df[TARGET]

X_test = test_df.drop(columns=DROP_COLS)
y_test = test_df[TARGET]

print(f"Train set: {X_train.shape[0]} samples ({train_end/n*100:.1f}%)")
print(f"Val set:   {X_val.shape[0]} samples ({(val_end-train_end)/n*100:.1f}%)")
print(f"Test set:  {X_test.shape[0]} samples ({(n-val_end)/n*100:.1f}%)")

# Verify temporal ordering
assert train_df['order_date'].max() <= val_df['order_date'].min(), "Train-Val overlap detected"
assert val_df['order_date'].max() <= test_df['order_date'].min(), "Val-Test overlap detected"
print("\n✓ Temporal ordering verified (no data leakage)")

Train set: 48948 samples (70.0%)
Val set:   10489 samples (15.0%)
Test set:  10489 samples (15.0%)

✓ Temporal ordering verified (no data leakage)


## 2. Baseline Evaluation: Linear Regression

In [8]:
# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predictions
lr_pred_train = lr_model.predict(X_train)
lr_pred_val = lr_model.predict(X_val)
lr_pred_test = lr_model.predict(X_test)

# Metrics
lr_results = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'MAE': [
        mean_absolute_error(y_train, lr_pred_train),
        mean_absolute_error(y_val, lr_pred_val),
        mean_absolute_error(y_test, lr_pred_test)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_train, lr_pred_train)),
        np.sqrt(mean_squared_error(y_val, lr_pred_val)),
        np.sqrt(mean_squared_error(y_test, lr_pred_test))
    ],
    'R²': [
        r2_score(y_train, lr_pred_train),
        r2_score(y_val, lr_pred_val),
        r2_score(y_test, lr_pred_test)
    ]
})

print("=" * 60)
print("LINEAR REGRESSION PERFORMANCE")
print("=" * 60)
print(lr_results.to_string(index=False))
print("\nTrain-Val Gap (MAE): {:.4f}h ({:.1f}%)".format(
    lr_results.loc[1, 'MAE'] - lr_results.loc[0, 'MAE'],
    (lr_results.loc[1, 'MAE'] - lr_results.loc[0, 'MAE']) / lr_results.loc[0, 'MAE'] * 100
))

ValueError: could not convert string to float: 'low'

### Linear Regression Analysis

In [ ]:
# Feature importance (coefficients)
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nTop 10 Most Important Features (by absolute coefficient):")
print(feature_importance.head(10).to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs Actual
axes[0].scatter(y_test, lr_pred_test, alpha=0.3, s=10)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Delivery Time (hours)')
axes[0].set_ylabel('Predicted Delivery Time (hours)')
axes[0].set_title('Linear Regression: Predicted vs Actual (Test Set)')
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = y_test - lr_pred_test
axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual (hours)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Residual Distribution (Mean: {residuals.mean():.3f}h)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. XGBoost Performance Analysis

In [ ]:
# Train XGBoost with champion configuration
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:absoluteerror',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

# Predictions
xgb_pred_train = xgb_model.predict(X_train)
xgb_pred_val = xgb_model.predict(X_val)
xgb_pred_test = xgb_model.predict(X_test)

# Metrics
xgb_results = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'MAE': [
        mean_absolute_error(y_train, xgb_pred_train),
        mean_absolute_error(y_val, xgb_pred_val),
        mean_absolute_error(y_test, xgb_pred_test)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_train, xgb_pred_train)),
        np.sqrt(mean_squared_error(y_val, xgb_pred_val)),
        np.sqrt(mean_squared_error(y_test, xgb_pred_test))
    ],
    'R²': [
        r2_score(y_train, xgb_pred_train),
        r2_score(y_val, xgb_pred_val),
        r2_score(y_test, xgb_pred_test)
    ]
})

print("=" * 60)
print("XGBOOST PERFORMANCE")
print("=" * 60)
print(xgb_results.to_string(index=False))
print("\nTrain-Val Gap (MAE): {:.4f}h ({:.1f}%)".format(
    xgb_results.loc[1, 'MAE'] - xgb_results.loc[0, 'MAE'],
    (xgb_results.loc[1, 'MAE'] - xgb_results.loc[0, 'MAE']) / xgb_results.loc[0, 'MAE'] * 100
))

### Direct Comparison: Linear Regression vs XGBoost

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'XGBoost'],
    'Train MAE': [lr_results.loc[0, 'MAE'], xgb_results.loc[0, 'MAE']],
    'Val MAE': [lr_results.loc[1, 'MAE'], xgb_results.loc[1, 'MAE']],
    'Test MAE': [lr_results.loc[2, 'MAE'], xgb_results.loc[2, 'MAE']],
    'Test RMSE': [lr_results.loc[2, 'RMSE'], xgb_results.loc[2, 'RMSE']],
    'Test R²': [lr_results.loc[2, 'R²'], xgb_results.loc[2, 'R²']]
})

print("=" * 80)
print("MODEL COMPARISON")
print("=" * 80)
print(comparison.to_string(index=False))

# Calculate improvement
mae_improvement = (lr_results.loc[2, 'MAE'] - xgb_results.loc[2, 'MAE']) / lr_results.loc[2, 'MAE'] * 100
print(f"\nXGBoost Test MAE Improvement: {mae_improvement:.2f}%")

## 4. Residual Diagnostics (XGBoost)

In [ ]:
# Calculate residuals
xgb_residuals_test = y_test.values - xgb_pred_test

print("=" * 60)
print("RESIDUAL DIAGNOSTICS (XGBoost Test Set)")
print("=" * 60)
print(f"Mean residual (bias): {xgb_residuals_test.mean():.4f}h")
print(f"Std residual (spread): {xgb_residuals_test.std():.4f}h")
print(f"Median residual: {np.median(xgb_residuals_test):.4f}h")
print(f"\nResidual range: [{xgb_residuals_test.min():.2f}, {xgb_residuals_test.max():.2f}]h")

# Percentiles
print("\nAbsolute Error Percentiles:")
abs_errors = np.abs(xgb_residuals_test)
for p in [50, 75, 90, 95, 99]:
    print(f"  {p}th percentile: {np.percentile(abs_errors, p):.3f}h")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Predicted Values (Heteroskedasticity Check)
axes[0, 0].scatter(xgb_pred_test, xgb_residuals_test, alpha=0.3, s=10)
axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Predicted Delivery Time (hours)')
axes[0, 0].set_ylabel('Residual (hours)')
axes[0, 0].set_title('Residuals vs Predicted Values')
axes[0, 0].grid(True, alpha=0.3)

# 2. Residual Distribution
axes[0, 1].hist(xgb_residuals_test, bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Residual (hours)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title(f'Residual Distribution (Mean: {xgb_residuals_test.mean():.3f}h)')
axes[0, 1].grid(True, alpha=0.3)

# 3. Absolute Error vs Actual Values (Error Magnitude Check)
axes[1, 0].scatter(y_test.values, abs_errors, alpha=0.3, s=10)
axes[1, 0].set_xlabel('Actual Delivery Time (hours)')
axes[1, 0].set_ylabel('Absolute Error (hours)')
axes[1, 0].set_title('Error Magnitude vs Target Value')
axes[1, 0].grid(True, alpha=0.3)

# 4. Q-Q Plot
from scipy import stats
stats.probplot(xgb_residuals_test, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot (Normality Check)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Segment-Level Stress Testing

In [ ]:
# Add predictions and errors to test dataframe
test_analysis = test_df.copy()
test_analysis['predicted'] = xgb_pred_test
test_analysis['absolute_error'] = abs_errors

print("=" * 60)
print("SEGMENT-LEVEL MAE ANALYSIS")
print("=" * 60)

In [ ]:
# 1. MAE by Traffic Level
if 'traffic_level' in test_analysis.columns:
    mae_by_traffic = test_analysis.groupby('traffic_level')['absolute_error'].agg(['mean', 'std', 'count'])
    mae_by_traffic.columns = ['MAE', 'Std', 'Count']
    print("\n1. Performance by Traffic Level:")
    print(mae_by_traffic.sort_values('MAE', ascending=False))
    
    # Visualization
    plt.figure(figsize=(10, 5))
    mae_by_traffic['MAE'].sort_values().plot(kind='barh', color='steelblue')
    plt.axvline(xgb_results.loc[2, 'MAE'], color='red', linestyle='--', label='Overall Test MAE')
    plt.xlabel('MAE (hours)')
    plt.title('XGBoost MAE by Traffic Level')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 2. MAE by Weather Condition
if 'weather' in test_analysis.columns:
    mae_by_weather = test_analysis.groupby('weather')['absolute_error'].agg(['mean', 'std', 'count'])
    mae_by_weather.columns = ['MAE', 'Std', 'Count']
    print("\n2. Performance by Weather Condition:")
    print(mae_by_weather.sort_values('MAE', ascending=False))
    
    # Visualization
    plt.figure(figsize=(10, 5))
    mae_by_weather['MAE'].sort_values().plot(kind='barh', color='coral')
    plt.axvline(xgb_results.loc[2, 'MAE'], color='red', linestyle='--', label='Overall Test MAE')
    plt.xlabel('MAE (hours)')
    plt.title('XGBoost MAE by Weather Condition')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 3. MAE by Vehicle Type
if 'vehicle_type' in test_analysis.columns:
    mae_by_vehicle = test_analysis.groupby('vehicle_type')['absolute_error'].agg(['mean', 'std', 'count'])
    mae_by_vehicle.columns = ['MAE', 'Std', 'Count']
    print("\n3. Performance by Vehicle Type:")
    print(mae_by_vehicle.sort_values('MAE', ascending=False))
    
    # Visualization
    plt.figure(figsize=(10, 5))
    mae_by_vehicle['MAE'].sort_values().plot(kind='barh', color='mediumseagreen')
    plt.axvline(xgb_results.loc[2, 'MAE'], color='red', linestyle='--', label='Overall Test MAE')
    plt.xlabel('MAE (hours)')
    plt.title('XGBoost MAE by Vehicle Type')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 4. MAE by Distance Buckets
if 'distance_km' in test_analysis.columns:
    test_analysis['distance_bucket'] = pd.qcut(test_analysis['distance_km'], q=4, 
                                                 labels=['Q1 (Short)', 'Q2 (Medium)', 'Q3 (Long)', 'Q4 (Very Long)'])
    mae_by_distance = test_analysis.groupby('distance_bucket')['absolute_error'].agg(['mean', 'std', 'count'])
    mae_by_distance.columns = ['MAE', 'Std', 'Count']
    print("\n4. Performance by Distance Bucket:")
    print(mae_by_distance)
    
    # Visualization
    plt.figure(figsize=(10, 5))
    mae_by_distance['MAE'].plot(kind='barh', color='mediumpurple')
    plt.axvline(xgb_results.loc[2, 'MAE'], color='red', linestyle='--', label='Overall Test MAE')
    plt.xlabel('MAE (hours)')
    plt.title('XGBoost MAE by Distance Bucket')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Summary: Evidence-Based Findings

**This section will be completed after running all cells above.**

Key observations to document:
1. Linear Regression performance baseline
2. XGBoost improvement magnitude
3. Train-validation gaps (overfitting evidence)
4. Residual patterns (bias, heteroskedasticity)
5. Weakest segments and domain explanations